# SDTP Lossless: Qwen3.8-27B (Multimodal Hybrid)

**Saliency-Protected Trit-Plane (SDTP)** — lossless-grade PTQ for the dense Qwen3.8 hybrid VL model.

Model: `Qwen/Qwen3.8-27B`  (architecture `Qwen3_5ForConditionalGeneration`)

| Stage | Technique | Output |
|---|---|---|
| **Setup** | Load BF16 on CPU, enumerate linears + skip list | ~54 GB BF16 |
| **Calib** | Activation-norm (saliency) capture on text | per-feature norms |
| **Pesos** | **SDTP Lossless** (Dual trit-plane @ 2% INT8 saliency) | ~4.4 bpw weights |
| **KV** | Hadamard-rotated + quantized KV (Q2/Q3) on the 16 full-attn layers | GDN state stays BF16 |
| **Visão** | Vision encoder kept in BF16 (SDTP not validated on vision) | intact |
| **Deploy** | Save + optional push to HF | `SDTP-Lossless` |

**Arquitetura**: 64 camadas densas (48 Gated DeltaNet linear-attn head_dim=128 + 16 Full Attention head_dim=256), vision-multimodal.
**Config**: `hidden=5120`, `intermediate=17408`, `num_q=24`, `num_kv=4`, `vocab=248320`, `ctx=262144`.

> SDTP = saliency protection (top-k% weights em INT8) + trit-plane asymétrico (per-row mu/alpha).
> Dual-plane dual = modulo lossless (~4.43 bpw). Mantém gates GDN (`in_proj_a/b`), embeddings, lm_head e encoder visual em BF16.


In [ ]:
# numpy/scipy first - reinstalados JUNTOS para ABI correta (scipy vs novo numpy)
!pip install -q --force-reinstall numpy==2.2.6 scipy
!pip install git+https://github.com/huggingface/transformers.git --force-reinstall -q
!pip install -q datasets accelerate safetensors sentencepiece tiktoken huggingface_hub tqdm requests Pillow

print('\n' + '='*60)
print('*** RESTART RUNTIME NOW (Runtime -> Restart session) ***')
print('Then run Cell 2 onwards. Do NOT re-run Cell 1 after restart.')
print('='*60)


In [ ]:
import torch, transformers, psutil
print(f'transformers: {transformers.__version__}')
print(f'torch:        {torch.__version__}')
print(f'CPU RAM:      {psutil.virtual_memory().total/1e9:.0f} GB')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'CUDA:         {p.name}, {p.total_memory/1e9:.1f} GB')
else:
    print('No CUDA detected - CPU-only mode')


In [ ]:
import torch, math, time, json, os, gc
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import norm as sp_norm
from collections import defaultdict

MODEL = 'Qwen/Qwen3.8-27B'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# GPU-aware: carrega na GPU se ela segurar o modelo BF16 (~54GB). RTX PRO 6000 = 96GB.
_AGB = torch.cuda.mem_get_info(0)[0]/1e9 if torch.cuda.is_available() else 0.0
GPU_OK = torch.cuda.is_available() and _AGB > 60.0
LOAD_DEV = 'cuda:0' if GPU_OK else 'cpu'
print(f'GPU disponível: {_AGB:.1f} GB -> GPU_OK={GPU_OK} (carrega em {LOAD_DEV})')
HEAD_DIM = 256
NUM_LAYERS = 64
LAYER_TYPES = ['gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn', 'gdn', 'gdn', 'gdn', 'full_attn']
assert len(LAYER_TYPES) == NUM_LAYERS
SAVE_DIR = '/content/sdtp_qwen38_27b'
os.makedirs(SAVE_DIR, exist_ok=True)

# ########## Walsh-Hadamard (para KV rotacionado) ##########
def _build_H(n):
    if n == 1:
        return torch.tensor([[1.0]])
    h = _build_H(n // 2)
    return torch.cat([
        torch.cat([h, h], dim=1),
        torch.cat([h, -h], dim=1),
    ], dim=0) / math.sqrt(2)

H256 = _build_H(HEAD_DIM)
print(f'H256: {tuple(H256.shape)}')

# ########## BitPacker (KV Q2/Q3/Q4) ##########
class BitPacker:
    @staticmethod
    def pack(codes, nbits):
        c = codes.long(); N = c.shape[0]
        if nbits == 2:
            c = c.reshape(N, -1, 4)
            return ((c[:,:,0]<<6)|(c[:,:,1]<<4)|(c[:,:,2]<<2)|c[:,:,3]).to(torch.uint8)
        elif nbits == 3:
            c = c.reshape(N, -1, 8)
            b0 = (c[:,:,0]<<5)|(c[:,:,1]<<2)|(c[:,:,2]>>1)
            b1 = ((c[:,:,2]&1)<<7)|(c[:,:,3]<<4)|(c[:,:,4]<<1)|(c[:,:,5]>>2)
            b2 = ((c[:,:,5]&3)<<6)|(c[:,:,6]<<3)|c[:,:,7]
            return torch.stack([b0, b1, b2], dim=-1).reshape(N, -1).to(torch.uint8)
        elif nbits == 4:
            return ((c[:,0::2]<<4)|c[:,1::2]).to(torch.uint8)
        return codes.to(torch.uint8)

    @staticmethod
    def unpack(packed, nbits, D):
        p = packed.long(); N = p.shape[0]
        if nbits == 2:
            return torch.stack([(p>>6)&3, (p>>4)&3, (p>>2)&3, p&3], dim=-1).reshape(N, D)
        elif nbits == 3:
            p3 = p.reshape(N, -1, 3)
            b0, b1, b2 = p3[:,:,0], p3[:,:,1], p3[:,:,2]
            return torch.stack([
                (b0>>5)&7, (b0>>2)&7, ((b0&3)<<1)|((b1>>7)&1),
                (b1>>4)&7, (b1>>1)&7, ((b1&1)<<2)|((b2>>6)&3),
                (b2>>3)&7, b2&7,
            ], dim=-1).reshape(N, D)
        elif nbits == 4:
            return torch.stack([(p>>4)&0xF, p&0xF], dim=-1).reshape(N, D)
        return p

# ########## PolarQuantLayer: KV rotacionado+quantizado (full-attn, head_dim=256) ##########
class PolarQuantLayer:
    def __init__(self, nbits=3, residual_length=128, device='cpu'):
        self.nbits = nbits; self.residual_length = residual_length; self.device = device
        self.H = H256.to(device); self.scale = math.sqrt(HEAD_DIM)
        self._packed = None; self._norms = None; self._q_seq = 0
        self._B = None; self._NH = None; self._D = None
        self._can_quantize = True; self.residual = None; self._ct = None

    def _quantize(self, tensor):
        flat = tensor.reshape(-1, HEAD_DIM).float()
        norms = flat.norm(dim=1, keepdim=True).clamp(min=1e-10)
        rotated = (flat / norms) @ self.H * self.scale
        if self._ct is None:
            self._build_ct()
        codes = (rotated.unsqueeze(-1) - self._ct.view(1, 1, -1)).abs().argmin(-1)
        return BitPacker.pack(codes.to(torch.uint8), self.nbits), norms.to(torch.bfloat16).squeeze(1)

    def _build_ct(self):
        n = 2**self.nbits
        lo = torch.linspace(-3.5, 3.5, n + 1)
        ct = (lo[:-1] + lo[1:]) / 2
        self._ct = ct.to(self.device)

    def _dequantize(self, packed, norms, B, H):
        codes = BitPacker.unpack(packed, self.nbits, HEAD_DIM)
        values = self._ct[codes] / self.scale
        values = (values @ self.H) * norms.float().unsqueeze(1)
        S = packed.shape[0] // (B * H)
        return values.to(torch.bfloat16).reshape(B, H, S, HEAD_DIM)

    def update(self, new_tensor):
        if self._B is None:
            self._B, self._NH = new_tensor.shape[0], new_tensor.shape[1]
            self._D = new_tensor.shape[3]
            self._can_quantize = (self._D == HEAD_DIM)
        self.residual = new_tensor if self.residual is None else torch.cat([self.residual, new_tensor], dim=2)
        if self._can_quantize and self.residual.shape[2] > self.residual_length:
            n_q = self.residual.shape[2] - self.residual_length
            packed, norms = self._quantize(self.residual[:, :, :n_q, :])
            self.residual = self.residual[:, :, n_q:, :].contiguous()
            if self._packed is None:
                self._packed, self._norms = packed, norms
            else:
                self._packed = torch.cat([self._packed, packed], dim=0)
                self._norms  = torch.cat([self._norms,  norms],  dim=0)
            self._q_seq += n_q
        if self._packed is not None:
            return torch.cat([self._dequantize(self._packed, self._norms, self._B, self._NH), self.residual], dim=2)
        return self.residual

    def get_seq_length(self):
        q = 0
        if self._packed is not None and self._B and self._NH:
            q = self._packed.shape[0] // (self._B * self._NH)
        return q + (self.residual.shape[2] if self.residual is not None else 0)

    def memory_bytes(self):
        t = 0
        if self._packed is not None: t += self._packed.numel() + self._norms.numel() * 2
        if self.residual is not None: t += self.residual.numel() * 2
        return t

# Hybrid cache layer (GDN linear attention states) - passthrough BF16
try:
    from transformers.cache_utils import LinearAttentionCacheLayerMixin
    _la_base = LinearAttentionCacheLayerMixin
except ImportError:
    _la_base = object

class _HybridCacheLayer(_la_base):
    def __init__(self):
        self.conv_states = None; self.recurrent_states = None
    def lazy_initialization(self, *args, **kwargs): pass
    def update_conv_state(self, conv_states, **kwargs):
        self.conv_states = conv_states; return conv_states
    def update_recurrent_state(self, recurrent_states, **kwargs):
        self.recurrent_states = recurrent_states; return recurrent_states

from transformers.cache_utils import Cache

class PolarQuantKVCache(Cache):
    def __init__(self, num_layers, nbits=3, residual_length=128, device='cpu'):
        try: super().__init__()
        except (TypeError, ValueError): pass
        self.num_layers = num_layers; self.nbits = nbits
        self.kl = [PolarQuantLayer(nbits, residual_length, device) for _ in range(num_layers)]
        self.vl = [PolarQuantLayer(nbits, residual_length, device) for _ in range(num_layers)]
        self._seen_tokens = 0
        if not hasattr(self, 'layers'): self.layers = [None] * num_layers

    def update(self, key_states, value_states, layer_idx, cache_kwargs=None):
        if layer_idx == 0: self._seen_tokens += key_states.shape[2]
        return self.kl[layer_idx].update(key_states), self.vl[layer_idx].update(value_states)

    def get_seq_length(self, layer_idx=0): return self.kl[layer_idx].get_seq_length()
    def get_max_cache_shape(self): return None
    def get_mask_sizes(self, query_length, layer_idx):
        return query_length, self.get_seq_length(layer_idx)
    def has_previous_state(self, layer_idx=0):
        has_kv = self.get_seq_length(layer_idx) > 0
        has_linear = self.layers[layer_idx] is not None and isinstance(self.layers[layer_idx], _HybridCacheLayer)
        return has_kv or has_linear
    def update_conv_state(self, conv_states, layer_idx, **kwargs):
        if self.layers[layer_idx] is None: self.layers[layer_idx] = _HybridCacheLayer()
        self.layers[layer_idx].conv_states = conv_states; return conv_states
    def update_recurrent_state(self, recurrent_states, layer_idx, **kwargs):
        if self.layers[layer_idx] is None: self.layers[layer_idx] = _HybridCacheLayer()
        self.layers[layer_idx].recurrent_states = recurrent_states; return recurrent_states
    @property
    def seen_tokens(self): return self._seen_tokens
    @property
    def is_initialized(self): return self._seen_tokens > 0
    def __getitem__(self, idx):
        k, v = self.kl[idx], self.vl[idx]
        k_out, v_out = k.residual, v.residual
        if k._packed is not None and k_out is not None:
            k_out = torch.cat([k._dequantize(k._packed, k._norms, k._B, k._NH), k_out], dim=2)
            v_out = torch.cat([v._dequantize(v._packed, v._norms, v._B, v._NH), v_out], dim=2)
        return (k_out, v_out)
    def __len__(self): return self.num_layers
    def __iter__(self):
        for i in range(self.num_layers): yield self[i]
    def memory_bytes(self):
        return sum(k.memory_bytes() + v.memory_bytes() for k, v in zip(self.kl, self.vl))

# ########## Skip patterns (SDTP) ##########
SKIP_PATTERNS = [
    'visual', 'vision', 'multi_modal_projector', 'patch_embed', 'patch_conv',
    'norm', 'layernorm', 'rmsnorm',
    'embed_tokens', 'lm_head',
    'gate', 'router',
    'A_log', 'conv1d', 'dt_bias',
    'linear_attn.in_proj_a', 'linear_attn.in_proj_b',
    'mtp',
]

def should_skip(name):
    for p in SKIP_PATTERNS:
        if p in name:
            return True
    return False

# ########## Tokenizer ##########
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
print(f'Tokenizer: {tokenizer.__class__.__name__}, vocab={tokenizer.vocab_size}')


In [ ]:
import torch, time
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM, AutoModel

print(f'Loading {MODEL} on {LOAD_DEV} (~54 GB BF16)...  (dtype=BF16)')
t_load = time.time()
model = None
for loader_name in ['AutoModelForImageTextToText', 'AutoModelForCausalLM', 'AutoModel']:
    try:
        loader = {'AutoModelForImageTextToText': AutoModelForImageTextToText,
                  'AutoModelForCausalLM': AutoModelForCausalLM,
                  'AutoModel': AutoModel}[loader_name]
        model = loader.from_pretrained(
            MODEL, dtype=torch.bfloat16, device_map=LOAD_DEV,
            trust_remote_code=True, low_cpu_mem_usage=True,
        )
        print(f'Loaded via {loader_name} on {LOAD_DEV}: {model.__class__.__name__}')
        break
    except Exception as e:
        print(f'  {loader_name} failed: {type(e).__name__}: {str(e)[:120]}')
if model is None: raise RuntimeError('All loaders failed')
print(f'Loaded in {time.time()-t_load:.0f}s')

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params/1e9:.2f}B ({total_params*2/1e9:.1f} GB BF16)')
arch_str = str(model.config.architectures or [model.__class__.__name__])
is_multimodal = 'ForConditionalGeneration' in arch_str or 'ImageTextToText' in arch_str
print(f'Architecture: {arch_str}, multimodal: {is_multimodal}')

# ########## Enumerate text Linear modules ##########
linear_modules = {}
skipped = []
for name, mod in model.named_modules():
    if not isinstance(mod, nn.Linear):
        continue
    if should_skip(name):
        skipped.append(name)
        continue
    linear_modules[name] = mod
print(f'Text linears to quantize: {len(linear_modules)}  (skipped: {len(skipped)})')
print('Sample skipped:', skipped[:12])

# ########## Optional brief category discovery ##########
from collections import Counter
cats = Counter('.'.join(n.split('.')[-2:]).rsplit('.',1)[0] for n in linear_modules)
print('\nTop module categories:'); [print(f'  {k:<55} {v}') for k, v in cats.most_common(15)]


In [ ]:
import numpy as np
from tqdm.auto import tqdm

# ########## Activation-norm capture (memory-efficient: sum-of-squares per feature) ##########
act_sumsq = {}

def make_hook(name):
    def hook(module, inp, outp):
        x = inp[0].detach().float()
        x_flat = x.reshape(-1, x.shape[-1])
        s = (x_flat ** 2).sum(dim=0).cpu().numpy()
        act_sumsq[name] = act_sumsq.get(name, np.zeros_like(s)) + s
    return hook

handles = [mod.register_forward_hook(make_hook(name)) for name, mod in linear_modules.items()]

calib_texts = [
    'Machine learning models use gradient descent to minimize a loss function over training data.',
    'The mitochondria is the powerhouse of the cell, generating ATP via oxidative phosphorylation.',
    'def fibonacci(n):\n    if n < 2:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)',
    'Quantum computers leverage superposition and entanglement to solve certain problems faster.',
    'A neural network maps inputs to outputs through stacked linear transformations and nonlinearities.',
    'import numpy as np\narr = np.array([1, 2, 3])\nprint(arr.mean())',
    'The history of mathematics spans Egyptian, Greek, Islamic and European contributions over millennia.',
    'Data compression reduces redundancy while preserving the information content of a signal.',
]

print('Calibrating activation norms (model on CPU, memory-efficient sum-of-squares)...')
eval_dev = LOAD_DEV
with torch.no_grad():
    for text in tqdm(calib_texts, desc='Calibration'):
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(eval_dev)
        try:
            model(**inputs)
        except Exception as e:
            print(f'  (skipping calib item: {type(e).__name__}: {str(e)[:80]})')

for h in handles: h.remove()
act_norms = {name: np.sqrt(v + 1e-10) for name, v in act_sumsq.items()}
act_sumsq.clear()
print(f'Captured norms for {len(act_norms)} modules')
gc.collect()


In [ ]:
import math
# ########## Baseline PPL (MUST run BEFORE quantization) ##########
# Reduzido p/ caber no Colab em CPU. Aumente os Ns numa máquina rápida.
N_CHUNKS = int(N_CHUNKS) if 'N_CHUNKS' in dir() else (24 if GPU_OK else 4)
MAX_LEN = int(MAX_LEN) if 'MAX_LEN' in dir() else (2048 if GPU_OK else 1024)

def load_eval_chunks():
    try:
        from datasets import load_dataset
        ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
        all_text = '\n'.join(x['text'] for x in ds if len(x['text'].strip()) > 50)
        full = tokenizer(all_text, return_tensors='pt').input_ids[0]
    except Exception as e:
        print('wikitext unavailable, using synthetic eval:', e)
        txt = ('. '.join(calib_texts * 20))
        full = tokenizer(txt, return_tensors='pt', truncation=True, max_length=4096).input_ids[0]
    chunks = []
    for i in range(0, len(full) - 1, MAX_LEN):
        c = full[i:i + MAX_LEN]
        if len(c) >= 64: chunks.append(c)
    return chunks

all_chunks = load_eval_chunks()
chunks = all_chunks[:N_CHUNKS]
print(f'Eval: {len(chunks)} chunks x {MAX_LEN} (from {len(all_chunks)} available) on {LOAD_DEV}')

def compute_ppl(chunks):
    total_loss = 0.0; total_tokens = 0; model.eval()
    with torch.no_grad():
        for chunk in tqdm(chunks, desc='PPL'):
            input_ids = chunk.unsqueeze(0).to(LOAD_DEV)
            if input_ids.shape[1] < 2: continue
            out = model(input_ids=input_ids, labels=input_ids)
            n = input_ids.shape[1] - 1
            total_loss += out.loss.item() * n
            total_tokens += n
    return math.exp(total_loss / total_tokens)

baseline_ppl = compute_ppl(chunks)
print(f'\n=== BASELINE PPL: {baseline_ppl:.4f} ===')


In [ ]:
import numpy as np, torch, time
from tqdm.auto import tqdm
from collections import defaultdict

# ########## SDTP core ##########
def optimal_ternary_offset(w):
    mu = w.mean()
    w_c = w - mu
    sigma = w_c.std()
    if sigma < 1e-10: return np.full_like(w, mu)
    tau = 0.612 * sigma
    alpha = 1.224 * sigma
    out = np.zeros_like(w)
    out[w_c > tau] = alpha
    out[w_c < -tau] = -alpha
    return out + mu

def sdtp_quantize(W, act_norm, outlier_frac=0.02, dual_plane=True):
    m, n = W.shape
    if act_norm is None or act_norm.shape[0] != n:
        act_norm = np.ones(n, dtype=np.float32)
    saliency = np.abs(W) * act_norm[None, :]
    n_out = max(1, int(outlier_frac * W.size))
    thr = np.partition(saliency.flatten(), -n_out)[-n_out]
    mask = saliency >= thr
    out = np.zeros_like(W)
    for i in range(m):
        row_mask = mask[i]
        if row_mask.any():
            vals = W[i][row_mask]
            mv = np.abs(vals).max() + 1e-10
            sc = mv / 127
            out[i, row_mask] = np.round(vals / sc).clip(-127, 127) * sc
        nsm = ~row_mask
        if nsm.any():
            vals = W[i][nsm]
            q1 = optimal_ternary_offset(vals)
            if dual_plane:
                q2 = optimal_ternary_offset(vals - q1)
                out[i, nsm] = q1 + q2
            else:
                out[i, nsm] = q1
    return out

OUTLIER_FRAC = 0.02
DUAL_PLANE = True

stats = defaultdict(lambda: {'count': 0, 'mse_sum': 0.0, 'params': 0})
quant_params = 0
print(f'Quantizing {len(linear_modules)} modules with SDTP Lossless (outlier={OUTLIER_FRAC*100:.0f}%, dual={DUAL_PLANE})...')
t0 = time.time()

for name, mod in tqdm(linear_modules.items(), desc='SDTP'):
    W = mod.weight.data.float().cpu().numpy()
    act_norm = act_norms.get(name)
    if act_norm is None: continue
    Wq = sdtp_quantize(W, act_norm, outlier_frac=OUTLIER_FRAC, dual_plane=DUAL_PLANE)
    mse = float(np.mean((W - Wq) ** 2))
    parts = name.split('.')
    key = '.'.join(p for p in parts[-2:] if not p.isdigit())
    stats[key]['count'] += 1
    stats[key]['mse_sum'] += mse * W.size
    stats[key]['params'] += W.size
    mod.weight.data = torch.from_numpy(Wq).to(dtype=mod.weight.dtype, device=mod.weight.device)
    quant_params += W.size
    del W, Wq

print(f'\nQuantized {quant_params/1e9:.2f}B params in {time.time()-t0:.0f}s')
print('\nPer-category average reconstruction MSE:')
worst = 0.0; best = float('inf')
for cat, s in sorted(stats.items(), key=lambda x: -x[1]['mse_sum']/max(1,x[1]['params'])):
    avg = s['mse_sum'] / max(1, s['params'])
    worst = max(worst, avg); best = min(best, avg)
    print(f'  {cat:<45} count={s["count"]:<4} avg_mse={avg:.6f}')
print(f'\nWorst/best MSE ratio: {worst/max(best,1e-12):.1f}x (want <5x)')


In [ ]:
# ########## Quantized PPL (same chunks as baseline) ##########
quantized_ppl = compute_ppl(chunks)
ratio = quantized_ppl / baseline_ppl
print(f'Baseline PPL:   {baseline_ppl:.4f}')
print(f'Quantized PPL:  {quantized_ppl:.4f}')
print(f'Ratio:          {ratio:.4f}x')
verdict = 'LOSSLESS (<=1.05x)' if ratio <= 1.05 else ('GOOD (<=1.5x)' if ratio <= 1.5 else 'BAD - check skip list/MSE')
print(f'Verdict: {verdict}')


In [ ]:
# ########## Text sanity (generation) ##########
prompts = [
    'The capital of France is',
    'def fibonacci(n):\n    if n < 2:\n        return n\n    return',
    'Explain why the sky is blue in one sentence:',
]
for p in prompts:
    inputs = tokenizer(p, return_tensors='pt').to(LOAD_DEV)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(out[0], skip_special_tokens=True))
    print('---')


In [ ]:
# ########## KV: Hadamard-rotated + quantized (16 full-attn layers), GDN state BF16 ##########
FULL_ATTN_IDX = [i for i, t in enumerate(LAYER_TYPES) if t == 'full_attn']
print('Full-attention layers:', FULL_ATTN_IDX)

N_TOKENS = 200
messages = [{'role': 'user', 'content': 'Write a detailed essay about the history of mathematics.'}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(text, return_tensors='pt').input_ids

def manual_generate(model, input_ids, cache, max_tokens):
    generated = []; next_tok = None; cur_cache = cache
    for step in range(max_tokens):
        with torch.no_grad():
            if step == 0:
                out = model(input_ids=input_ids, past_key_values=cur_cache, use_cache=True)
            else:
                out = model(input_ids=next_tok, past_key_values=cur_cache, use_cache=True)
        cur_cache = out.past_key_values
        next_tok = out.logits[:, -1:].argmax(-1)
        tid = next_tok.item(); generated.append(tid)
        if tid in (tokenizer.eos_token_id, getattr(tokenizer, 'pad_token_id', None)): break
    return generated, cur_cache

model.eval()

t0 = time.time()
fp16_tokens, _ = manual_generate(model, input_ids, None, N_TOKENS)
fp16_time = time.time() - t0

pq3_cache = PolarQuantKVCache(NUM_LAYERS, nbits=3, residual_length=128, device='cpu')
t0 = time.time()
pq3_tokens, pq3_out = manual_generate(model, input_ids, pq3_cache, N_TOKENS)
pq3_time = time.time() - t0
kv_bytes = pq3_out.memory_bytes() if hasattr(pq3_out, 'memory_bytes') else 0

print(f'FP16 KV: {len(fp16_tokens)} tok in {fp16_time:.1f}s ({len(fp16_tokens)/max(fp16_time,1e-6):.1f} tok/s)')
print(f'Q3 KV:   {len(pq3_tokens)} tok in {pq3_time:.1f}s ({len(pq3_tokens)/max(pq3_time,1e-6):.1f} tok/s)')
print(f'Q3 KV memory: {kv_bytes/1e6:.1f} MB')
min_len = min(len(fp16_tokens), len(pq3_tokens))
matches = sum(1 for a, b in zip(fp16_tokens[:min_len], pq3_tokens[:min_len]) if a == b)
print(f'Token match: {matches}/{min_len} ({100*matches/max(min_len,1):.1f}%)')


In [ ]:
# ########## Quality showcase (text) ##########
prompts = [
    'Write a Python function that implements binary search. Include type hints.',
    'Explain TCP vs UDP in simple terms.',
    'What causes the aurora borealis? Explain the physics briefly.',
]
for i, prompt in enumerate(prompts, 1):
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        out = model.generate(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask,
                             max_new_tokens=250, do_sample=True, temperature=0.7, top_p=0.9,
                             pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f'\nPrompt {i}: {prompt}\n{resp}\n{"-"*60}')


In [ ]:
# ########## SDTP nao validado em visao: encoder visual mantido em BF16. Validar visao intacta. ##########
def try_vision():
    try:
        from transformers import AutoProcessor
        import requests
        from PIL import Image
        proc = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
        url = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png'
        img = Image.open(requests.get(url, stream=True, timeout=30).raw).convert('RGB')
        messages = [{'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text', 'text': 'Describe this image in one sentence.'},
        ]}]
        text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = proc(text=[text], images=[img], return_tensors='pt')
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=60, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        print('Vision OK:', proc.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
    except Exception as e:
        print(f'Vision validation skipped/failed: {type(e).__name__}: {str(e)[:160]}')
try_vision()


In [ ]:
import os
print('=== SDTP Lossless Summary ===')
print(f'Model: Qwen/Qwen3.8-27B')
print(f'Architecture: {model.__class__.__name__} (dense 27B, 48 GDN + 16 full attn)')
print(f'Config: hidden 5120, FFN 17408, head_dim {HEAD_DIM} (full), vocab 248320')
print()
print(f'=== SDTP Lossless (Dual @ 2%) ===')
print(f'  Quantized: {quant_params/1e9:.2f}B text params')
print(f'  Bits/weight: ~4.43 (dual trit-plane + saliency INT8)')
print(f'  Baseline PPL:  {baseline_ppl:.3f}')
print(f'  Quantized PPL: {quantized_ppl:.3f}')
print(f'  Ratio:         {ratio:.3f}x')
print()
print('=== KV (16 full-attn layers) ===')
print(f'  Rotated (Hadamard) + quantized Q2/Q3, residual 128')
print(f'  GDN linear state: BF16 (passthrough)  Vision encoder: BF16')
print()
print('=== Estimated size (text weights) ===')
print(f'  BF16: ~{total_params*2/1e9:.1f} GB | SDTP @4.4bpw: ~{total_params*0.55/1e9:.1f} GB (text)')
print(f'  Vision encoder mantido em BF16 (adicional ~2-4 GB)')


In [ ]:
import torch, json, os, time

# ########## Save reconstructed model (BF16 weights, vision intact) ##########
print('Saving full model state (BF16 reconstruction) ...')
SAVE_MODEL = f'{SAVE_DIR}/model_tensors'
model.save_pretrained(SAVE_MODEL)
tokenizer.save_pretrained(SAVE_DIR)

# ########## Save calibration + stats ##########
with open(f'{SAVE_DIR}/sdtp_stats.json', 'w') as f:
    json.dump({
        'method': 'SDTP-Lossless',
        'outlier_frac': OUTLIER_FRAC, 'dual_plane': DUAL_PLANE,
        'bits_per_weight_est': 4.43,
        'baseline_ppl': baseline_ppl, 'quantized_ppl': quantized_ppl, 'ratio': ratio,
    }, f, indent=2)
print('Saved stats + config')

# ########## Upload (opcional) ##########
HF_TOKEN = ''  # @param {type:'string'} -- cole seu token do HF
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import HfApi
    api = HfApi()
    repo_id = 'caiovicentino1/Qwen3.8-27B-SDTP-Lossless'
    api.create_repo(repo_id, repo_type='model', exist_ok=True, private=False)
    api.upload_large_folder(folder_path=SAVE_DIR, repo_id=repo_id, repo_type='model')
    print(f'Uploaded: https://huggingface.co/caiovicentino1/Qwen3.8-27B-SDTP-Lossless')
else:
    print('[!] HF_TOKEN vazio - defina acima e re-execute esta celula para subir')
